In [144]:
import pickle
import numpy as np
import random
from music21 import *
import os
# 读取存储的转移矩阵字典
with open('transition_matrices.pkl', 'rb') as f:
    transition_matrices = pickle.load(f)
    
# 读取存储的转移矩阵字典
with open('mean_tempo.pkl', 'rb') as f:
    mean_tempo = pickle.load(f)
    
from pyAudioAnalysis import audioTrainTest as aT
from midi2audio import FluidSynth
import os
os.environ["PATH"] += os.pathsep + r"E:/TOOL SOFT/fluidsynth/bin"

In [150]:
def musicgenerate(place,level,month):
    # 指定参数
    key = f'{place}_{level}_{month}'
    
    # 获取对应的马尔可夫矩阵
    if key in transition_matrices:
        transition_df = transition_matrices[key]
        
        # 获取转移概率矩阵和状态列表
        transition_probabilities = transition_df.copy()
        
        # 归一化转移概率矩阵
        transition_probabilities = transition_probabilities.div(transition_probabilities.sum(axis=1), axis=0)
        
        # 清理矩阵：将小于 0.05 的值设置为 0
        transition_probabilities[transition_probabilities < 0.05] = 0
        
        # 获取和弦名称列表
        chord_names = transition_probabilities.columns.tolist()
        
        # 转换为 NumPy 数组
        markov_matrix = transition_probabilities.values
        
        # 选择起始和弦
        current_chord = random.choice(chord_names)
        generated_sequence = [current_chord]
    
        # 生成和弦序列
        sequence_length = 100  # 设定序列的长度
        for i in range(sequence_length - 1):
            # 每间隔5个和弦，进行重启逻辑
            if i > 0 and i % 5 == 0:
                # 获取未曾出现过的和弦
                unseen_chords = [chord for chord in chord_names if chord not in generated_sequence]
                if unseen_chords:
                    current_chord = random.choice(unseen_chords)
                else:
                    # 如果所有和弦都出现过，则选择出现次数较少的和弦
                    chord_counts = {chord: generated_sequence.count(chord) for chord in chord_names}
                    min_count = min(chord_counts.values())
                    least_seen_chords = [chord for chord, count in chord_counts.items() if count == min_count]
                    current_chord = random.choice(least_seen_chords)
            
            current_index = chord_names.index(current_chord)
            
            # 获取当前和弦的转移概率
            probabilities = markov_matrix[current_index].copy()
            
            # 降低选择上一个和弦的概率
            if len(generated_sequence) > 1:
                previous_chord = generated_sequence[-2]
                if previous_chord in chord_names:
                    previous_index = chord_names.index(previous_chord)
                    probabilities[previous_index] *= 0.5
            
            # 降低选择刚使用过和弦的概率
            last_chord = generated_sequence[-1]
            if last_chord in chord_names:
                last_index = chord_names.index(last_chord)
                probabilities[last_index] *= 0.5
            
            # 正常化概率，确保总和为1
            probabilities_sum = probabilities.sum()
            if probabilities_sum > 0:
                probabilities /= probabilities_sum
            else:
                probabilities = markov_matrix[current_index]  # 恢复原始概率
            
            # 选择下一个和弦
            next_chord = random.choices(chord_names, weights=probabilities)[0]
            generated_sequence.append(next_chord)
            current_chord = next_chord
    
        # 输出生成的和弦序列
        print(f"Generated chord sequence for {key}:")
        print(" -> ".join(generated_sequence))
    else:
        print(f"No data found for key: {key}")
    
    # 定义和弦性质的映射
    quality_mapping = {
        "maj": ["P1", "M3", "P5"],  # 大三和弦
        "min": ["P1", "m3", "P5"],  # 小三和弦
        "dim": ["P1", "m3", "d5"],  # 减三和弦
        "aug": ["P1", "M3", "A5"],  # 增三和弦
        # 可以继续添加其他和弦类型
    }
    
    # 定义和弦序列
    chord_sequence = generated_sequence
    
    tempo_mean = mean_tempo.get(f'{place}_{level}_{month}_mean', 120)
    tempo_std = mean_tempo.get(f'{place}_{level}_{month}_std', 10)
    
    # 创建一个音乐流对象
    s = stream.Stream()
    
    # 初始化旋律部分
    melody = stream.Part()
    
    # 初始化节奏
    current_tempo = np.random.normal(loc=tempo_mean, scale=tempo_std)
    quarter_note_duration = 60 / current_tempo  # 将BPM转换为每个四分音符的时长
    
    # 设置和弦部分
    chords = stream.Part()
    
    # 初始化用于控制感情变化的力度
    base_velocity = 70
    
    # 为每个和弦生成旋律和和弦部分
    for i, chord_name in enumerate(chord_sequence):
        root, quality = chord_name.split(':')
        intervals = quality_mapping.get(quality, ["P1", "M3", "P5"])
        pitches = [pitch.Pitch(root).transpose(interval.Interval(i)) for i in intervals]
        
        # 生成旋律音符，确保与和弦节奏一致
        melody_notes = []
        for j in range(4):
            note_pitch = random.choice(pitches)
            n = note.Note(note_pitch)
            n.quarterLength = quarter_note_duration  # 使用统一的四分音符时长
            n.volume.velocity = base_velocity + random.randint(-10, 10)  # 引入动态变化
            melody_notes.append(n)
            n.offset = 0
            melody.append(n)
    
        # 创建和弦并添加到和弦部分
        c = chord.Chord(pitches)
        c.quarterLength = 4 * quarter_note_duration  # 和弦持续四个四分音符时长
        chords.append(c)
    
        # 每10个和弦后更新一次节奏
        if (i + 1) % 10 == 0:
            current_tempo = np.random.normal(loc=tempo_mean, scale=tempo_std)
            quarter_note_duration = 60 / current_tempo
    
    # 将旋律和和弦部分添加到音乐流中
    s.append(melody)
    s.append(chords)

    
    # 设置文件路径
    # 创建输出文件夹
    output_folder = 'musicgenerate'
    if not os.path.exists(output_folder):
        os.makedirs(output_folder)
    
    file_base_name = f"{place}_{level}_{month}music"
    midi_fp = os.path.join(output_folder, f"{file_base_name}.mid")
    wav_fp = os.path.join(output_folder, f"{file_base_name}.wav")
    txt_fp = os.path.join(output_folder, f"{file_base_name}.txt")
    # 保存为 MIDI 文件
    s.write('midi', fp=midi_fp)
    fs = FluidSynth(sound_font=r'F:\作业文件\BVOC-SOA\CI\sounddata\数据库处理\GeneralUser\GeneralUser GS v1.471.sf2')
    fs.midi_to_audio(midi_fp, wav_fp)
     # 保存和弦序列到文本文件
    with open(txt_fp, 'w') as txt_file:
        txt_file.write(" -> ".join(generated_sequence))

In [156]:
all_combinations = [key.split('_') for key in transition_matrices.keys()]
# 循环遍历每一个组合并调用函数
for combination in all_combinations:
    place, level, month = combination
    success = False
    while not success:
        try:
            musicgenerate(place, level, month)
            success = True  # 成功后退出循环
        except TranslateException:
            print(f"Error encountered for {place}_{level}_{month}, retrying...")

Generated chord sequence for CM_16 m_12:
E:maj -> F#:maj -> G#:maj -> E:maj -> C:maj -> G#:maj -> E:maj -> F#:maj -> G:min -> G:min -> G:min -> C:maj -> A#:maj -> E:maj -> D:maj -> F#:maj -> A:maj -> A:maj -> F#:maj -> D:maj -> E:maj -> G#:maj -> G#:maj -> G#:maj -> G#:maj -> G#:maj -> G#:maj -> G#:maj -> G#:maj -> E:maj -> D:maj -> G#:maj -> G#:maj -> E:maj -> C:maj -> G#:maj -> G#:maj -> E:maj -> D:maj -> F#:maj -> G:min -> G#:maj -> G#:maj -> G#:maj -> G#:maj -> G#:maj -> E:maj -> F#:maj -> D:maj -> E:maj -> F#:maj -> G#:maj -> G#:maj -> G#:maj -> E:maj -> F#:maj -> D:maj -> E:maj -> D:maj -> E:maj -> F#:maj -> D:maj -> E:maj -> F#:maj -> D:maj -> E:maj -> G#:maj -> G#:maj -> G#:maj -> G#:maj -> E:maj -> A:maj -> F#:maj -> G:min -> G:min -> G:min -> C:maj -> G#:maj -> G#:maj -> E:maj -> F#:maj -> G#:maj -> G#:maj -> G#:maj -> G#:maj -> G#:maj -> F#:maj -> D:maj -> E:maj -> F#:maj -> G:min -> D#:maj -> G#:maj -> G#:maj -> G#:maj -> G#:maj -> D:maj -> E:maj -> D:maj -> E:maj
Generated

In [ ]:


# # 指定参数
# place = 'CM'  
# level = '1.5 m'  
# month = '9'  
# key = f'{place}_{level}_{month}'
# 
# # 获取对应的马尔可夫矩阵
# if key in transition_matrices:
#     transition_df = transition_matrices[key]
#     
#     # 获取转移概率矩阵和状态列表
#     transition_probabilities = transition_df.copy()
#     
#     # 归一化转移概率矩阵
#     transition_probabilities = transition_probabilities.div(transition_probabilities.sum(axis=1), axis=0)
#     
#     # 清理矩阵：将小于 0.05 的值设置为 0
#     transition_probabilities[transition_probabilities < 0.05] = 0
#     
#     # 获取和弦名称列表
#     chord_names = transition_probabilities.columns.tolist()
#     
#     # 转换为 NumPy 数组
#     markov_matrix = transition_probabilities.values
#     
#     # 选择起始和弦
#     current_chord = random.choice(chord_names)
#     generated_sequence = [current_chord]
# 
#     # 生成和弦序列
#     sequence_length = 20  # 设定序列的长度
#     for _ in range(sequence_length - 1):
#         current_index = chord_names.index(current_chord)
#         next_chord = random.choices(chord_names, weights=markov_matrix[current_index])[0]
#         generated_sequence.append(next_chord)
#         current_chord = next_chord
# 
#     # 输出生成的和弦序列
#     print(f"Generated chord sequence for {key}:")
#     print(" -> ".join(generated_sequence))
# else:
#     print(f"No data found for key: {key}")


In [96]:
# # 创建一个音乐流对象
# s = stream.Stream()
# 
# # 创建旋律部分
# melody = stream.Part()
# 
# # 为每个和弦生成旋律
# for chord_name in chord_sequence:
#     root, quality = chord_name.split(':')
#     # 获取和弦的音程
#     intervals = quality_mapping.get(quality, ["P1", "M3", "P5"])  # 获取完整的和弦性质的音程
#     pitches = [pitch.Pitch(root).transpose(interval.Interval(i)) for i in intervals]
#     c = chord.Chord(pitches)
# 
#     # 生成旋律音符
#     for _ in range(4):  # 为每个和弦生成 4 个音符
#         note_pitch = random.choice(c.pitches)  # 从和弦音符中随机选择
#         n = note.Note(note_pitch)
#         n.quarterLength = 0.5  # 每个音符持续半个四分音符
#         melody.append(n)
# 
# # 将旋律部分添加到音乐流中
# s.append(melody)
# 
# # 创建和弦部分
# chords = stream.Part()
# for chord_name in chord_sequence:
#     root, quality = chord_name.split(':')
#     # 获取和弦的音程
#     intervals = quality_mapping.get(quality, ["P1", "M3", "P5"])  # 获取完整的和弦性质的音程
#     pitches = [pitch.Pitch(root).transpose(interval.Interval(i)) for i in intervals]
#     c = chord.Chord(pitches)
#     c.quarterLength = 2  # 每个和弦持续两个四分音符
#     chords.append(c)
# 
# # 将和弦部分添加到音乐流中
# s.append(chords)
# 
# # 保存为 MIDI 文件
# midi_fp = 'melody_chord_progression.mid'
# s.write('midi', fp=midi_fp)

'melody_chord_progression.mid'